# C7-cnn-transfer — Practice p26 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** convolution, tensor-shape-tracing, cnn-training

The helper carries the output channels and the two independently updated spatial dimensions forward after each specification. For the large input its trace is `(3,31,96,227) → (3,43,32,114) → (3,59,32,36)`. The smaller stack preserves the helper-led order: its trace is committed before any torch layer is built, and only then is it checked against constructed outputs and trained.

## Part I — pure-Python floor-formula tracer

In [ ]:
def trace_conv_stack(input_shape, specs):
    batch, channels, height, width = input_shape
    traced_shapes = []
    for out_channels, kernel_h, kernel_w, stride_h, stride_w, pad_h, pad_w in specs:
        height = (height + 2 * pad_h - kernel_h) // stride_h + 1
        width = (width + 2 * pad_w - kernel_w) // stride_w + 1
        if height < 1 or width < 1:
            raise ValueError("convolution produces a spatial size below one")
        channels = out_channels
        traced_shapes.append((batch, channels, height, width))
    return traced_shapes


input_shape = (3, 17, 191, 227)
specs = [
    (31, 7, 3, 2, 1, 3, 1),
    (43, 3, 5, 3, 2, 0, 2),
    (59, 1, 7, 1, 3, 0, 0),
]
trace = trace_conv_stack(input_shape, specs)

## Part II — helper-led construction and training

In [ ]:
import torch
from torch import nn

SEED = 20260804
torch.set_default_dtype(torch.float64)
torch.manual_seed(SEED)
small_input_shape = (15, 2, 13, 15)
small_specs = [
    (4, 3, 5, 1, 2, 1, 2),
    (6, 3, 3, 2, 1, 1, 1),
]
small_trace = trace_conv_stack(small_input_shape, small_specs)
if not isinstance(small_trace, list) or len(small_trace) != len(small_specs):
    raise RuntimeError("commit a complete helper-produced small_trace first")

generator = torch.Generator(device="cpu").manual_seed(SEED)
train_X = 0.05 * torch.randn(*small_input_shape, generator=generator)
train_y = torch.arange(small_input_shape[0], dtype=torch.long) % 3
train_X[train_y == 0, :, :, 2:5] += 1.0
train_X[train_y == 1, :, 6:9, :] += 1.0
diagonal = torch.arange(13)
class_two = train_X[train_y == 2].clone()
class_two[:, :, diagonal, diagonal] += 1.0
train_X[train_y == 2] = class_two
train_X[train_y == 0] -= 0.8
train_X[train_y == 2] += 0.8

In [ ]:
features = nn.Sequential(
    nn.Conv2d(2, 4, kernel_size=(3, 5), stride=(1, 2), padding=(1, 2)),
    nn.ReLU(),
    nn.Conv2d(4, 6, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1)),
    nn.ReLU(),
)
model = nn.Sequential(
    features,
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(1),
    nn.Linear(6, 3),
)

constructed_shapes = []
with torch.no_grad():
    traced_value = train_X
    for layer in features:
        traced_value = layer(traced_value)
        if isinstance(layer, nn.Conv2d):
            constructed_shapes.append(tuple(traced_value.shape))
constructed_trace_agrees = constructed_shapes == small_trace

parameter_before = {
    name: parameter.detach().clone() for name, parameter in model.named_parameters()
}
optimizer = torch.optim.SGD(model.parameters(), lr=0.12)
criterion = nn.CrossEntropyLoss()
loss_history = []
for _ in range(20):
    optimizer.zero_grad(set_to_none=True)
    logits = model(train_X)
    loss = criterion(logits, train_y)
    loss_history.append(float(loss.detach()))
    loss.backward()
    optimizer.step()

model_parameter_ids = {id(parameter) for parameter in model.parameters()}
optimizer_parameter_objects = [
    parameter for group in optimizer.param_groups for parameter in group["params"]
]
optimizer_parameter_ids = {id(parameter) for parameter in optimizer_parameter_objects}
optimizer_owns_exactly_model = (
    optimizer_parameter_ids == model_parameter_ids
    and len(optimizer_parameter_objects) == len(list(model.parameters()))
)
gradient_names = sorted(
    name for name, parameter in model.named_parameters() if parameter.grad is not None
)
moved_parameter_names = sorted(
    name
    for name, parameter in model.named_parameters()
    if not torch.equal(parameter.detach(), parameter_before[name])
)
training_certificate = bool(
    constructed_trace_agrees
    and tuple(logits.shape) == (15, 3)
    and optimizer_owns_exactly_model
    and gradient_names == sorted(name for name, _ in model.named_parameters())
    and moved_parameter_names
    and len(loss_history) == 20
    and torch.isfinite(torch.tensor(loss_history)).all()
    and loss_history[-1] <= 0.75 * loss_history[0]
)

### Answer check

In [ ]:
assert trace == [
    (3, 31, 96, 227),
    (3, 43, 32, 114),
    (3, 59, 32, 36),
]
assert small_trace == [(15, 4, 13, 8), (15, 6, 7, 8)]
assert trace_conv_stack((1, 2, 5, 5), [(3, 3, 3, 1, 1, 0, 0)]) == [(1, 3, 3, 3)]
invalid_size_rejected = False
try:
    trace_conv_stack((1, 2, 2, 2), [(3, 5, 5, 1, 1, 0, 0)])
except ValueError:
    invalid_size_rejected = True
assert invalid_size_rejected
assert constructed_shapes == small_trace and constructed_trace_agrees
assert isinstance(features[0], nn.Conv2d) and isinstance(features[1], nn.ReLU)
assert isinstance(features[2], nn.Conv2d) and isinstance(features[3], nn.ReLU)
assert (features[0].in_channels, features[0].out_channels) == (2, 4)
assert features[0].kernel_size == (3, 5) and features[0].stride == (1, 2) and features[0].padding == (1, 2)
assert (features[2].in_channels, features[2].out_channels) == (4, 6)
assert features[2].kernel_size == (3, 3) and features[2].stride == (2, 1) and features[2].padding == (1, 1)
assert isinstance(model[1], nn.AdaptiveAvgPool2d) and model[1].output_size == (1, 1)
assert isinstance(model[2], nn.Flatten) and model[2].start_dim == 1
assert isinstance(model[3], nn.Linear) and (model[3].in_features, model[3].out_features) == (6, 3)
assert tuple(logits.shape) == (15, 3)
assert optimizer_parameter_ids == {id(parameter) for parameter in model.parameters()}
assert len(optimizer_parameter_objects) == len(list(model.parameters()))
assert optimizer_owns_exactly_model
assert gradient_names == sorted(name for name, _ in model.named_parameters())
assert moved_parameter_names
assert set(moved_parameter_names).issubset(dict(model.named_parameters()))
assert len(loss_history) == 20
assert torch.isfinite(torch.tensor(loss_history)).all()
assert loss_history[-1] <= 0.75 * loss_history[0]
assert training_certificate